In [ ]:
import pandas as pd
import requests
import sys 
import time
# Load data
game_frame = pd.read_csv(
    "https://raw.githubusercontent.com/gabriel1200/shot_data/refs/heads/master/game_dates.csv"
)

# Filter seasons
game_frame = game_frame[game_frame.season > "2023-24"]
games = game_frame[["GAME_ID", "season", "playoffs"]].drop_duplicates()

# Request headers
headers = {
    "Host": "stats.nba.com",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:72.0) Gecko/20100101 Firefox/72.0",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Referer": "https://stats.nba.com/"
}

def extract_video_urls(data):
    """
    Parses the JSON data to extract all video 'surl' values.

    Args:
        data (dict): The parsed JSON data from the API.

    Returns:
        list: A list of all the video 'surl' strings.
    """
    video_urls = []
    try:
        # Navigate through the dictionary to the list of video URLs
        video_details_list = data['resultSets']['Meta']['videoUrls']

        # Loop through each video detail dictionary in the list
        for video in video_details_list:
            # Check if 'surl' key exists and its value is not None
            if 'surl' in video and video['surl'] is not None:
                video_urls.append(video['surl'])
    except KeyError as e:
        print(f"Error: The key {e} was not found in the JSON data.")
    except TypeError as e:
        print(f"Error: The data structure was not as expected. {e}")
        
    return video_urls
# Loop over games
all_links = set()

for _, gamerow in games.iterrows():
    game_id = str(gamerow["GAME_ID"])
    season = gamerow["season"]
    stype = "Playoffs" if gamerow["playoffs"] else "Regular Season"

    print(game_id, season, stype)
    contexts=["FGA","AST","STL","TOV","REB","BLK"]
    for context in contexts:
      

        url = (
            f"https://stats.nba.com/stats/videodetailsasset?"
            f"AheadBehind=&CFID=&CFPARAMS=&ClutchTime=&Conference=&ContextFilter=&ContextMeasure={context}"
            f"&DateFrom=&DateTo=&Division=&EndPeriod=0&EndRange=28800&GROUP_ID="
            f"&GameEventID=&GameID=00{game_id}&GameSegment=&GroupID=&GroupMode=&GroupQuantity=5"
            f"&LastNGames=0&LeagueID=00&Location=&Month=0&OnOff=&OppPlayerID=&OpponentTeamID=0"
            f"&Outcome=&PORound=0&Period=0&PlayerID=0&PlayerID1=&PlayerID2=&PlayerID3=&PlayerID4="
            f"&PlayerID5=&PlayerPosition=&PointDiff=&Position=&RangeType=0&RookieYear=&Season={season}"
            f"&SeasonSegment=&SeasonType={stype}&ShotClockRange=&StartPeriod=0&StartRange=0&StarterBench="
            f"&TeamID=0&VsConference=&VsDivision=&VsPlayerID1=&VsPlayerID2=&VsPlayerID3=&VsPlayerID4="
            f"&VsPlayerID5=&VsTeamID="
        )

        response = requests.get(url, headers=headers)
        response.raise_for_status()  # throws error if request failed
        data = response.json()
        links = extract_video_urls(data)
        all_links.update(links)  # add unique links
        print(len(links))
     
        time.sleep(2)
df_links = pd.DataFrame(sorted(all_links), columns=["link"])
df_links.to_csv("saved_urls.csv", index=False)


IndentationError: unexpected indent (3309741020.py, line 88)